# WETH/USDT LP의 IL · LVR · Predictable Loss

이 노트북은 `non_same_block_pair_returns.parquet`의 10,795개 multi-block operation pair를 동일한 표본으로 사용한다. 핵심 공식과 event-order 계산은 `analysis/src/lp_risk`에 있으며, 노트북은 versioned output을 생성하고 표와 차트를 표시한다.

- 평가통화: USDT, 외부가격: event 직전 1초 Binance ETHUSDT
- pool 경로: `(block_number, transaction_index, log_index)` 순서의 exact `sqrt_price_x96`; 주 PL은 block-end mesh
- IL/LVR/PL: fee와 gas 제외
- risk-free rate: 동결된 FRED SOFR snapshot, ACT/360
- PL은 Predictable Loss이며 profit and loss가 아니다.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
from IPython.display import display

matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

REPOSITORY_ROOT = Path(".").resolve()
sys.path.insert(0, str(REPOSITORY_ROOT / "data" / "src"))
sys.path.insert(0, str(REPOSITORY_ROOT / "analysis" / "src"))

from lp_risk.pipeline import build_risk_metrics
from uniswap_v3_data.paths import resolve_data_root

DATA_ROOT = resolve_data_root()
OUTPUT_ROOT = DATA_ROOT / "derived" / "risk_metrics" / "v2"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)


## 1. 재현 가능한 output 생성

노트북 실행 중에는 외부 네트워크를 사용하지 않는다. SOFR snapshot이 없다면 먼저 `python analysis/scripts/prepare_sofr.py`를 한 번 실행한다. 이후 실행은 입력 manifest와 고정된 snapshot만 읽는다.


In [ ]:
artifacts = build_risk_metrics(
    DATA_ROOT, output_root=OUTPUT_ROOT, repository_root=REPOSITORY_ROOT
)
position_metrics = pd.read_parquet(artifacts.position_metrics)
portfolio_daily = pd.read_parquet(artifacts.portfolio_daily)
representatives = pd.read_parquet(artifacts.representative_positions)
representative_paths = pd.read_parquet(artifacts.representative_paths)
run_manifest = json.loads(artifacts.manifest.read_text())

display(pd.DataFrame({
    "item": ["positions", "strict sensitivity positions", "position-days", "representative path rows"],
    "count": [
        len(position_metrics),
        int(position_metrics["is_strict_pair"].sum()),
        int(portfolio_daily["position_day_count"].sum()),
        len(representative_paths),
    ],
}))
display(pd.DataFrame(run_manifest["artifacts"]))


## 2. 공식과 부호

$P_t$는 내부 pool 가격, $S_t$는 외부 Binance 가격, $x(P),y(P)$는 fee-exclusive inventory다. $V(P)=x(P)P+y(P)$, $W(P,S)=x(P)S+y(P)$, $H(S)=x_0S+y_0$로 둔다. 전체 정의와 유도는 `analysis/RISK_METRICS.md`에 고정한다.

### Impermanent Loss

$$IL_t^{signed}=\frac{V_t^{LP}-V_t^{HODL}}{V_t^{HODL}}.$$

Entry와 exit은 실제 Mint/Burn amount, 중간 경로는 exact pool sqrt-price에서 복원한 fee-exclusive inventory를 사용한다. 비교 차트에는 $L_t^{IL}=[H(S_t)-W(P_t,S_t)]/V_0$를 loss-positive 값으로 표시한다. 정확한 이산 dynamics는 $\Delta L_j^{IL}=x_0\Delta S_j-[W(P_j,S_j)-W(P_{j-1},S_{j-1})]$이며, 이 증분은 양·음 모두 가능하다.

### Loss-versus-Rebalancing

$$\Delta LVR_k=-\left(S_k\Delta x_k+\Delta y_k\right).$$

여기서 $S_k$는 Swap 직전 Binance 가격이다. 실증 LVR은 pool–CEX basis와 이산적 event timing 때문에 감소하거나 음수일 수 있어 clipping하지 않는다. CEX QV의 1초·5초·1분 근사는 sensitivity로만 둔다.

### Predictable Loss

보유 중 Swap이 있는 block을 $b=1,\ldots,B$로 두고, $\bar P_0$는 entry 직전 상태, $\bar P_b$는 block $b$의 마지막 in-scope Swap 후 상태로 둔다. 주지표는

$$C_b=V(\bar P_{b-1})+x(\bar P_{b-1})(\bar P_b-\bar P_{b-1})-V(\bar P_b),$$
$$D_b=G(t_{b-1},t_b)D_{b-1}+C_b.$$

논문 용어를 따라 **PL = Convexity Cost + Opportunity Cost**로 명명한다. `pl_convexity_cost`는 $\sum C_b$, `pl_opportunity_cost`는 $D_T-\sum C_b$, 논문 부호의 PL은 $-D_T$다. 모든 Swap 상태를 쓰는 이전 방식은 `pl_swap_event_*` sensitivity로만 보존한다. SOFR는 전체 WETH inventory가 아니라 이미 발생한 replication gap에만 적용된다. 따라서 Convexity Cost가 전혀 없으면 $r>0$이어도 PL은 0이다.

### 연속시간 dynamics (이상적 $P=S$ limit)

$$dV_t^{LP}=x(S_t)dS_t+\frac12V''(S_t)d[S]_t,$$
$$dL_t^{IL}=(x_0-x(S_t))dS_t-\frac12V''(S_t)d[S]_t,$$
$$dLVR_t=dC_t=-\frac12V''(S_t)d[S]_t,$$
$$dD_t=dC_t+r_tD_tdt,\qquad dO_t=r_tD_tdt.$$

따라서 $dL^{IL}=(x_0-x)dS+dLVR$이고, $r=0$의 이상적 limit에서는 $dPL_{loss}=dLVR=dC$다. 실제 $P\ne S$ 데이터에서는 이 식을 직접 강제하지 않고 위의 exact discrete increments를 누적한다.


In [ ]:
distribution_columns = [
    "holding_seconds",
    "il_loss_on_initial",
    "lvr_rebalancing_on_initial",
    "lvr_qv_1s_on_initial",
    "lvr_qv_5s_on_initial",
    "lvr_qv_1m_on_initial",
    "pl_convexity_cost_on_initial",
    "pl_opportunity_cost_on_initial",
    "pl_loss_on_initial",
    "pl_swap_event_loss_on_initial",
]
display(position_metrics[distribution_columns].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T)


## 3. Capital-weighted multi-block operation-pair risk metrics

각 날짜와 half-open lifetime이 겹치는 position을 `min(exit, UTC day-end)`에서 평가하고, dollar loss 합계를 initial capital 합계로 나눈 사후 position-day 대표값이다. 표본 진입·청산에 따라 선이 점프할 수 있으며 실시간 투자 portfolio가 아니다.


In [ ]:
daily = portfolio_daily.sort_values("date")
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True,
                         gridspec_kw={"height_ratios": [1.0, 0.7, 1.7, 0.7]})
axes[0].plot(daily["date"], daily["ethusdt_close"], color="#333333", linewidth=1.0)
axes[0].set(title="ETHUSDT daily close", ylabel="USDT per ETH")
axes[1].plot(daily["date"], daily["sofr_percent"], color="#E45756", linewidth=1.1)
axes[1].set(title="Frozen FRED SOFR (calendar-day forward-fill)", ylabel="Percent p.a.")
axes[2].plot(daily["date"], daily["capital_weighted_il_loss_pct"], label="IL", linewidth=1.1)
axes[2].plot(daily["date"], daily["capital_weighted_lvr_loss_pct"], label="LVR (external rebalancing)", linewidth=1.1)
axes[2].plot(daily["date"], daily["capital_weighted_pl_loss_pct"], label="Predictable Loss", linewidth=1.2)
axes[2].plot(daily["date"], daily["capital_weighted_pl_convexity_cost_pct"], label="Convexity Cost", linestyle="--", linewidth=0.9, alpha=0.8)
axes[2].fill_between(daily["date"], daily["capital_weighted_pl_convexity_cost_pct"], daily["capital_weighted_pl_loss_pct"], color="#E45756", alpha=0.15, label="Opportunity Cost")
axes[2].set(title="Capital-weighted multi-block operation-pair risk metrics (loss-positive)", ylabel="Percent of initial capital")
axes[2].legend(loc="best", ncol=2)
axes[3].step(daily["date"], daily["position_day_count"], where="mid", color="#7A5195")
axes[3].set(title="Position-day count", ylabel="Positions", xlabel="Date (UTC)")
fig.tight_layout()
fig.savefig(FIGURE_ROOT / "capital_weighted_risk_metrics.png", dpi=180, bbox_inches="tight")
plt.show()


## 4. 사후 시장방향 × lifetime 대표 position

상승·하락은 각 position 보유기간 ETH log return의 상·하위 25%, short·long은 전체 holding time의 하·상위 25%다. 각 교집합에서 initial capital, range width, lifetime, ETH return의 robust medoid를 고른다. 이 분류는 미래 exit 가격을 쓰는 설명용 사례이며 confirmatory regime 변수가 아니다.


In [ ]:
representative_table = representatives.copy()
representative_table["holding_days"] = representative_table["holding_seconds"] / 86_400
representative_table["eth_return_pct"] = 100 * np.expm1(representative_table["eth_log_return"])
display(representative_table[[
    "market_regime", "lifetime_bucket", "candidate_count", "operation_id",
    "entry_timestamp", "exit_timestamp", "holding_days", "eth_return_pct",
    "initial_wealth_usdt", "tick_lower", "tick_upper", "is_strict_pair",
]])


In [ ]:
def plot_representative_position(metadata, path):
    kind_rank = path["row_kind"].map({"entry": 0, "swap": 1, "grid": 2, "exit": 3})
    path = path.assign(_kind_rank=kind_rank).sort_values(["timestamp", "_kind_rank", "block_number", "transaction_index", "log_index"], kind="stable", na_position="last").drop(columns="_kind_rank")
    holding_days = float(metadata["holding_seconds"]) / 86_400
    eth_return = 100 * np.expm1(float(metadata["eth_log_return"]))
    initial_capital = float(metadata["initial_wealth_usdt"])
    label = f"{metadata['market_regime'].title()} / {metadata['lifetime_bucket'].title()}"
    fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True, gridspec_kw={"height_ratios": [1.0, 1.0, 1.35]})
    axes[0].plot(path["timestamp"], path["external_ethusdt"], color="#333333", linewidth=0.9, label="Binance ETHUSDT")
    axes[0].axhline(path["range_lower_usdt"].iloc[0], color="#4C78A8", linestyle="--", linewidth=0.8, label="LP range")
    axes[0].axhline(path["range_upper_usdt"].iloc[0], color="#4C78A8", linestyle="--", linewidth=0.8)
    axes[0].set(title=f"{label}: ETH price and LP range", ylabel="USDT per ETH")
    axes[0].legend(loc="best")
    axes[1].plot(path["timestamp"], path["lp_intrinsic_value_usdt"], label="LP intrinsic value (fee-exclusive)", linewidth=1.0)
    axes[1].plot(path["timestamp"], path["hodl_value_usdt"], label="HODL benchmark", linewidth=1.0)
    axes[1].set(title="Position value", ylabel="USDT")
    axes[1].legend(loc="best")
    axes[2].plot(path["timestamp"], 100 * path["il_loss_on_initial"], label="IL", linewidth=1.0)
    axes[2].plot(path["timestamp"], 100 * path["lvr_rebalancing_on_initial"], label="LVR", linewidth=1.0)
    axes[2].plot(path["timestamp"], 100 * path["pl_loss_on_initial"], label="Predictable Loss", linewidth=1.1)
    axes[2].plot(path["timestamp"], 100 * path["pl_swap_event_loss_on_initial"], label="Swap-event PL sensitivity", color="#999999", linestyle=":", linewidth=0.8, alpha=0.8)
    axes[2].plot(path["timestamp"], 100 * path["pl_convexity_cost_usdt"] / initial_capital, label="Convexity Cost", linestyle="--", linewidth=0.9)
    axes[2].fill_between(path["timestamp"], 100 * path["pl_convexity_cost_usdt"] / initial_capital, 100 * path["pl_loss_on_initial"], color="#E45756", alpha=0.15, label="Opportunity Cost")
    axes[2].set(title="Cumulative risk measures (loss-positive)", ylabel="Percent of initial capital", xlabel="Timestamp (UTC)")
    axes[2].legend(loc="best", ncol=2)
    fig.suptitle(f"{metadata['operation_id'][:12]}… | {holding_days:.2f} days | ETH {eth_return:+.2f}% | initial {initial_capital:,.0f} USDT", y=1.01)
    fig.tight_layout()
    filename = f"representative_{metadata['market_regime']}_{metadata['lifetime_bucket']}.png"
    fig.savefig(FIGURE_ROOT / filename, dpi=180, bbox_inches="tight")
    plt.show()

for _, metadata in representatives.iterrows():
    path = representative_paths.loc[representative_paths["operation_id"] == metadata["operation_id"]].copy()
    plot_representative_position(metadata, path)


## 5. 해석 시 주의점

- `lvr_rebalancing`은 외부 Binance 가격에서 같은 inventory를 self-financing으로 조정하는 benchmark다. pool–CEX basis 때문에 실증 경로가 항상 단조증가하지는 않는다.
- 주 Convexity Cost는 block별 마지막 in-scope pool 상태를 잇는다. 한 block 안의 급락·복귀를 별도의 hedge 가능한 가격변화로 중복 계상하지 않는다.
- 모든 Swap을 쓰는 `pl_swap_event_*`는 microstructure/mesh sensitivity다. transient intra-block excursion 때문에 주 PL보다 크게 spike할 수 있다.
- SOFR opportunity component는 이미 발생한 PL gap의 이자이며 전체 LP principal 또는 WETH inventory의 대체수익이 아니다.
- QV LVR와 `expected_pl_r0_30d`는 강한 연속시간·가격일치·변동성 가정이 필요한 robustness 지표다.
- 대표 position의 bull/bear label은 exit까지의 가격을 사용한 사후 설명용 구분이다.
